In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split

In [2]:
REPO_ID = "YihuaiHong/ConceptVectors"
SOURCE  = "ConceptVectors_data/llama2-7b_concepts.json"
STORE   = Path("../store/conceptvectors")
STORE.mkdir(parents=True, exist_ok=True)

PREFIX = "Say something about: "
MIN_SENTENCE_CHARS = 20
MIN_CONCEPT_ROWS = 100  # drop concepts with fewer rows than this before balancing
SENT_RE = re.compile(r"(?<=[.!?])\s+")

In [3]:
# Source format per row:
#   Concept, QA (list[str]), wikipedia_content (str), text_completion (list[{First_half, Second_half}])
# We flatten everything into (concept, question, answer) where:
#   QA questions used as-is (only ~10 per concept; too few alone)
#   wikipedia_content split into sentences, prefixed with PREFIX
#   text_completion halves joined per pair, sentence-split, prefixed with PREFIX
#   answer left blank (the dataset has no answers)
# Some concept names repeat (with different probe layer/dim); their rows merge.
# Concepts with very little flattened content get dropped before balancing.

def split_sentences(text: str) -> list[str]:
    text = text.replace("\n", " ").strip()
    return [s.strip() for s in SENT_RE.split(text) if len(s.strip()) >= MIN_SENTENCE_CHARS]

path = hf_hub_download(REPO_ID, SOURCE, repo_type="dataset")
with open(path) as f:
    data = json.load(f)

rows = []
for row in data:
    concept = row["Concept"]
    for q in row.get("QA", []):
        rows.append({"concept": concept, "question": q.strip(), "answer": ""})
    for s in split_sentences(row.get("wikipedia_content", "")):
        rows.append({"concept": concept, "question": PREFIX + s, "answer": ""})
    for tc in row.get("text_completion", []):
        combined = (tc.get("First_half", "") + " " + tc.get("Second_half", "")).strip()
        for s in split_sentences(combined):
            rows.append({"concept": concept, "question": PREFIX + s, "answer": ""})

df = pd.DataFrame(rows)
df["question"] = df["question"].str.strip()
df = df[df["question"] != ""].drop_duplicates(subset=["concept", "question"]).reset_index(drop=True)

sizes = df.groupby("concept").size()
keep = sizes[sizes >= MIN_CONCEPT_ROWS].index
df = df[df["concept"].isin(keep)].reset_index(drop=True)

min_n = df.groupby("concept").size().min()
balanced = pd.concat(g.sample(min_n, random_state=42) for _, g in df.groupby("concept")).reset_index(drop=True)
train_df, test_df = train_test_split(balanced, test_size=0.2, random_state=42, stratify=balanced["concept"])

train_df.to_csv(STORE / "train.csv", index=False)
test_df.to_csv(STORE / "test.csv", index=False)

def stats(df):
    by = df.groupby("concept").size()
    return {"rows": len(df), "concepts": by.shape[0], "rows/concept (mean)": round(by.mean(), 1), "min-max": f"{by.min()}-{by.max()}"}

print(pd.DataFrame({"train": stats(train_df), "test": stats(test_df)}).T)
print(f"\ntrain/test ratio: {len(train_df)/len(test_df):.2f}")

       rows concepts rows/concept (mean) min-max
train  7644       91                84.0   84-84
test   1911       91                21.0   21-21

train/test ratio: 4.00


In [4]:
print("concepts:")
for c in sorted(train_df["concept"].unique()):
    print(" ", c)

concepts:
  Amazon Alexa
  Amazon Kindle
  Amazon Prime
  American Civil War
  American football
  Ancient Rome
  Array (data structure)
  Artificial intelligence
  Astrological sign
  Austria
  Baseball
  Bayesian inference
  Bible
  Blockchain
  CT scan
  Cannabis
  Cantonese
  Catholic Church
  Chinese characters
  Civil rights movement
  Columbine High School massacre
  Communism
  Cowboy
  Cricket
  Culture of Africa
  Culture of Greece
  Culture of Italy
  Culture of Latin America
  Diabetes
  Disney Princess
  Egypt
  England
  Euro
  Figure skating
  Formula One
  Freemasonry
  Genetic engineering
  Germany
  Golf
  Greece
  HTTP cookie
  Halloween
  Harry Potter
  Heroin
  History of the Netherlands
  India
  International Red Cross and Red Crescent Movement
  Islam
  Islamic State
  Italy
  Jesus
  Jewish culture
  Jules Verne
  Julius Caesar
  Korea
  London
  Los Angeles
  Mafia
  Martial arts
  Marvel Comics
  McDonald's
  National Basketball Association
  National Health 